In [1]:
from transcription_pipeline.RateExtraction import AverageAndFit
import pandas as pd
import matplotlib as mpl
mpl.use('TkAgg')

In [2]:
dataset_folder = '/mnt/Data1/Nick/transcription_pipeline/'
key = '003'

embryo_list = {
    '001': [
        # 'test_data/NSPARC/2025-08-29/MCP-mSG_His-RFP_RBS(001)(trans-het)_embryo01',
        'test_data/NSPARC/2025-08-29/MCP-mSG_His-RFP_RBS(001)(trans-het)_embryo02',
        'test_data/NSPARC/2025-09-02/MCP-mSG_His-RFP_RBS(001)(trans-het)_embryo01',
        'test_data/NSPARC/2025-09-09/MCP-mSG_His-RFP_RBS(001)(trans-het)_embryo01',
        'test_data/NSPARC/2025-09-12/MCP-mSG_His-RFP_RBS(001)(trans-het)_embryo01',

    ],

    '002': [
        # 'test_data/NSPARC/2025-08-28/MCP-mSG_His-RFP_RBS(002)(trans-het)_embryo01',
        # 'test_data/NSPARC/2025-08-28/MCP-mSG_His-RFP_RBS(002)(trans-het)_embryo02',
        'test_data/NSPARC/2025-08-28/MCP-mSG_His-RFP_RBS(002)(trans-het)_embryo03',
        'test_data/NSPARC/2025-09-12/MCP-mSG_His-RFP_RBS(002)(trans-het)_embryo01',
        'test_data/NSPARC/2025-09-12/MCP-mSG_His-RFP_RBS(002)(trans-het)_embryo02',
        'test_data/NSPARC/2025-09-19/MCP-mSG_His-RFP_RBS(002)(trans-het)_embryo02',
        'test_data/NSPARC/2025-09-19/MCP-mSG_His-RFP_RBS(002)(trans-het)_embryo03',
    ],

    '003': [
        'test_data/NSPARC/2025-08-28/MCP-mSG_His-RFP_RBS(003)(trans-het)_embryo01',
        'test_data/NSPARC/2025-09-02/MCP-mSG_His-RFP_RBS(003)(trans-het)_embryo01',
        'test_data/NSPARC/2025-09-02/MCP-mSG_His-RFP_RBS(003)(trans-het)_embryo02',
        'test_data/NSPARC/2025-09-02/MCP-mSG_His-RFP_RBS(003)(trans-het)_embryo03',
        # 'test_data/NSPARC/2025-09-09/MCP-mSG_His-RFP_RBS(003)(trans-het)_embryo01',
        'test_data/NSPARC/2025-09-09/MCP-mSG_His-RFP_RBS(003)(trans-het)_embryo02',
    ]
}
test_dataset_name = dataset_folder + embryo_list[key][4]
print('Dataset Path: ' + test_dataset_name)

Dataset Path: /mnt/Data1/Nick/transcription_pipeline/test_data/NSPARC/2025-09-09/MCP-mSG_His-RFP_RBS(003)(trans-het)_embryo02


In [3]:
# Specify here at what frame NC14 starts
nc14_start_frame = 0

# Width of averaging window for time bins
time_bin_width = 2

# Number of bins you want to split the full embryo into
num_bins = 40

# Load in compiled dataframe
dataframe_path = test_dataset_name + '/compiled_dataframe.pkl'
compiled_dataframe = pd.read_pickle(dataframe_path)

In [4]:
compiled_dataframe.head()

,particle,frame,t_s,intensity_from_neighborhood,intensity_std_error_from_neighborhood,x,y,ap,ap90
0,2,"[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13,...","[3.98360995000042, 6.180576950000599, 8.203596...","[125.32950128534704, 102.60135904255318, 232.4...","[52.41592137737932, 53.72569001527491, 52.3012...","[902.6370668081356, 901.6625446374533, 900.393...","[161.4636837286949, 160.82794787927955, 160.78...","[0.18865799982340783, 0.18901773933954027, 0.1...","[-0.05035218815274012, -0.04949994237155536, -..."
1,3,"[21, 23, 25, 28, 29, 30, 31, 32, 33, 34, 35, 3...","[46.466733950000254, 50.51266294999979, 54.558...","[75.92147368421053, 184.38511444141687, 85.061...","[55.11321681875425, 50.518444291379744, 58.979...","[870.4041605152887, 870.6381293412527, 871.313...","[187.26549590578847, 188.13325149621159, 188.3...","[0.2039045138828859, 0.20386927130577173, 0.20...","[-0.07184610009727135, -0.07280940908811825, -..."
2,4,"[379, 380, 381, 383, 384, 385, 386, 387, 389, ...","[771.7675519500002, 773.7905069500003, 775.813...","[220.0815808219178, 139.366, 107.6088194070080...","[56.477666871954476, 56.37889715132783, 58.926...","[806.5849515545887, 807.027810614601, 808.0028...","[120.12011781661579, 120.56273116996962, 121.4...","[0.2256390832314969, 0.22548659338230018, 0.22...","[0.01103248827258064, 0.010482119148105214, 0...."
3,5,"[373, 379, 380, 381, 382, 383, 384, 385, 386, ...","[758.7375559500009, 771.2455679499991, 773.442...","[216.1455517241379, 242.28740163934427, 253.63...","[48.63966841815007, 50.86968414465683, 47.9287...","[958.7993830759057, 958.8948809142584, 959.608...","[161.67719996786803, 161.95491442878014, 161.9...","[0.1653219376042532, 0.16530208562441248, 0.16...","[-0.060823974822461796, -0.06113603133622653, ..."
4,6,"[341, 342, 346, 347, 350, 351, 354, 357, 358, ...","[694.7193769499995, 696.5683759499993, 704.660...","[67.18959349593497, 177.19799421965317, 242.19...","[59.847767926272816, 59.23194008882531, 59.015...","[810.2737303101827, 811.0090902298431, 810.094...","[196.88868630086637, 197.11332071308465, 197.5...","[0.22959369628116372, 0.22930400583015503, 0.2...","[-0.07108649699385057, -0.07145896443224964, -..."


In [5]:
aafdata = AverageAndFit(compiled_dataframe, nc14_start_frame, time_bin_width, num_bins, test_dataset_name)

No previous bin fit checking results detected. Do bin fitting for the dataframe.


/mnt/Data1/Nick/transcription_pipeline/transcription_pipeline/RateExtraction.py:1437: RuntimeWarning: Mean of empty slice.
  time_bin_means = np.array([intensity_flat[time_bin_indices == i].mean() for i in range(1, len(time_bins))])
/mnt/Data1/Nick/miniforge3/envs/transcription_pipeline/lib/python3.10/site-packages/numpy/core/_methods.py:129: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)
/mnt/Data1/Nick/miniforge3/envs/transcription_pipeline/lib/python3.10/site-packages/numpy/core/_methods.py:206: RuntimeWarning: Degrees of freedom <= 0 for slice
  ret = _var(a, axis=axis, dtype=dtype, out=out, ddof=ddof,
/mnt/Data1/Nick/miniforge3/envs/transcription_pipeline/lib/python3.10/site-packages/numpy/core/_methods.py:163: RuntimeWarning: invalid value encountered in divide
  arrmean = um.true_divide(arrmean, div, out=arrmean,
/mnt/Data1/Nick/miniforge3/envs/transcription_pipeline/lib/python3.10/site-packages/numpy/core/_methods.py:198: Runtime

Failed to fit average trace 18: All standard errors must be positive
Failed to fit average trace 19: All standard errors must be positive


/mnt/Data1/Nick/transcription_pipeline/transcription_pipeline/RateExtraction.py:1488: FutureWarning: ChainedAssignmentError: behaviour will change in pandas 3.0!
You are setting values through chained assignment. Currently this works in certain cases, but when using Copy-on-Write (which will become the default behaviour in pandas 3.0) this will never work to update the original DataFrame or Series, because the intermediate object on which we are setting values will behave as a copy.
A typical example is when you are setting values in a column of a DataFrame, like:

df["col"][row_indexer] = value

Use `df.loc[row_indexer, "col"] = values` instead, to perform the assignment in a single step and ensure this keeps updating the original `df`.

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy

  self.bin_average_fit_dataframe['bin_fit_result'][bin] = bin_fit_result
/mnt/Data1/Nick/transcription_pipeline

In [6]:
mpl.use('TkAgg')
aafdata.check_bin_fits()

Moved to bin 6 out of 40, the first unchecked bin


In [7]:
aafdata.save_checked_bin_fits()

Checked bin fits saved


In [8]:
aafdata.plot_bin_fits()

(array([0.   , 0.025, 0.05 , 0.075, 0.1  , 0.125, 0.15 , 0.175, 0.2  ,
        0.225, 0.25 , 0.275, 0.3  , 0.325, 0.35 , 0.375, 0.4  , 0.425,
        0.45 , 0.475, 0.5  , 0.525, 0.55 , 0.575, 0.6  , 0.625, 0.65 ,
        0.675, 0.7  , 0.725, 0.75 , 0.775, 0.8  , 0.825, 0.85 , 0.875,
        0.9  , 0.925, 0.95 , 0.975]),
 array([         nan,          nan,          nan,          nan,
                 nan,  99.38323665, 125.96458318, 155.13228068,
        188.0423643 , 182.66402324, 146.62684298, 195.23743825,
        167.70362157, 148.25439332, 107.84249407, 121.69163439,
        157.82992444,          nan,          nan,          nan,
                 nan,          nan,          nan,          nan,
                 nan,          nan,          nan,          nan,
                 nan,          nan,          nan,          nan,
                 nan,          nan,          nan,          nan,
                 nan,          nan,          nan,          nan]),
 array([[         nan,          nan,

In [ ]:
# Load checked particle fits
fits_path = test_dataset_name + '/bin_fits_checked.pkl'
bin_fits_checked = pd.read_pickle(fits_path)